In [ ]:
%%sql

-- Granularidad:
--   1 detección NASA × 1 carretera OSM
-- Espacial:
--   carretera <=25 km del incendio
-- DGT:
--   incidencia próxima a la carretera
--   y temporalmente relacionada con el incendio


DROP TABLE IF EXISTS gold_fire_road_impact;

--creacion de la tabla gold para analizar el impacto de los incendios en la red viaria
CREATE TABLE gold_fire_road_impact (

    fire_detection_id STRING,
    cluster_id STRING,
    fire_detection_timestamp TIMESTAMP,
    fire_latitude DOUBLE,
    fire_longitude DOUBLE,
    fire_radiative_power DOUBLE,
    osm_way_id STRING,
    road_reference STRING,
    road_name STRING,
    road_classification STRING,
    road_latitude DOUBLE,
    road_longitude DOUBLE,
    distance_fire_road_km DOUBLE,
    max_speed_kmh INT,
    lanes_count INT,
    is_oneway STRING,
    pavement_surface STRING,
    has_bridge STRING,
    has_tunnel STRING,
    traffic_incident_count BIGINT,
    active_traffic_incident_count BIGINT,
    max_traffic_severity STRING,
    road_impact_level STRING,
    fire_source_file STRING,
    osm_source_file STRING,
    gold_updated_at TIMESTAMP
);


--NASA

CREATE OR REPLACE TEMP VIEW road_fire_fires AS

SELECT

    SHA2(
        CONCAT_WS(
            '|',
            CAST(latitude AS STRING),
            CAST(longitude AS STRING),
            CAST(fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_detection_id,
    cluster_id,
    fire_detection_timestamp,
    CAST(latitude AS DOUBLE) AS fire_latitude,
    CAST(longitude AS DOUBLE) AS fire_longitude,
    CAST(fire_radiative_power AS DOUBLE)
        AS fire_radiative_power,
    landing_source_file AS fire_source_file

FROM silver_nasa_fires

WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND fire_detection_timestamp IS NOT NULL;


--OSM

--vista para formatear tramos viarios de osm con sus centroides y atributos fisicos
CREATE OR REPLACE TEMP VIEW road_data AS

SELECT

    CAST(osm_way_id AS STRING) AS osm_way_id,
    road_reference,
    road_name,
    road_classification,
    CAST(centroid_latitude AS DOUBLE)
        AS road_latitude,
    CAST(centroid_longitude AS DOUBLE)
        AS road_longitude,
    CAST(max_speed_kmh AS INT)
        AS max_speed_kmh,
    CAST(lanes_count AS INT)
        AS lanes_count,
    is_oneway,
    pavement_surface,
    has_bridge,
    has_tunnel,
    landing_source_file AS osm_source_file

FROM silver_osm_roads

WHERE osm_way_id IS NOT NULL
  AND centroid_latitude IS NOT NULL
  AND centroid_longitude IS NOT NULL;


--Distancia incendio-carretera

--he aplicado un prefiltro por deltas geofraficas antes de ejecutar el calculo de haversine
CREATE OR REPLACE TEMP VIEW fire_road_candidates AS

SELECT

    f.fire_detection_id,
    f.cluster_id,
    f.fire_detection_timestamp,
    f.fire_latitude,
    f.fire_longitude,
    f.fire_radiative_power,
    r.osm_way_id,
    r.road_reference,
    r.road_name,
    r.road_classification,
    r.road_latitude,
    r.road_longitude,
    r.max_speed_kmh,
    r.lanes_count,
    r.is_oneway,
    r.pavement_surface,
    r.has_bridge,
    r.has_tunnel,
    r.osm_source_file,
    (
        6371.0 * 2.0 * ASIN(
            SQRT(
                POWER(
                    SIN(
                        RADIANS(
                            r.road_latitude
                            - f.fire_latitude
                        ) / 2.0
                    ),
                    2
                )
                +
                COS(
                    RADIANS(f.fire_latitude)
                )
                *
                COS(
                    RADIANS(r.road_latitude)
                )
                *
                POWER(
                    SIN(
                        RADIANS(
                            r.road_longitude
                            - f.fire_longitude
                        ) / 2.0
                    ),
                    2
                )
            )
        )
    ) AS distance_fire_road_km,
    f.fire_source_file

FROM road_fire_fires f

INNER JOIN road_data r

    ON r.road_latitude BETWEEN
        f.fire_latitude - 0.25
        AND
        f.fire_latitude + 0.25
   AND r.road_longitude BETWEEN
        f.fire_longitude - 0.35
        AND
        f.fire_longitude + 0.35;

--Carreteras <=25 km

--filtrado de las carreteras situadas dentro del radio maximo de 25 kilometros
CREATE OR REPLACE TEMP VIEW fire_road_matches AS

SELECT *

FROM fire_road_candidates

WHERE distance_fire_road_km <= 25.0;


--DGT asociada a la carretera

--cruce de incidencias de trafico dgt cercanas con ventana temporal de tres horas
CREATE OR REPLACE TEMP VIEW road_traffic AS

SELECT

    f.fire_detection_id,
    f.osm_way_id,
    COUNT(
        DISTINCT d.record_id
    ) AS traffic_incident_count,
    COUNT(
        DISTINCT CASE
            WHEN d.start_timestamp
                    <= f.fire_detection_timestamp
             AND (
                    d.end_timestamp IS NULL
                    OR
                    d.end_timestamp
                        >= f.fire_detection_timestamp
                 )
            THEN d.record_id
        END
    ) AS active_traffic_incident_count,
    CASE
        WHEN MAX(
            CASE
                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) IN ('CRITICAL', 'SEVERE')
                THEN 4
                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'HIGH'
                THEN 3
                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'MEDIUM'
                THEN 2
                ELSE 1
            END
        ) = 4
            THEN 'CRITICAL'
        WHEN MAX(
            CASE
                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'HIGH'
                THEN 3
                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'MEDIUM'
                THEN 2
                ELSE 1
            END
        ) = 3
            THEN 'HIGH'
        WHEN MAX(
            CASE
                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'MEDIUM'
                THEN 2
                ELSE 1
            END
        ) = 2
            THEN 'MEDIUM'
        ELSE 'NORMAL'
    END AS max_traffic_severity
FROM fire_road_matches f

LEFT JOIN silver_dgt_traffic d

    ON d.latitude IS NOT NULL
   AND d.longitude IS NOT NULL
   AND d.latitude BETWEEN
        f.road_latitude - 0.05
        AND
        f.road_latitude + 0.05
   AND d.longitude BETWEEN
        f.road_longitude - 0.07
        AND
        f.road_longitude + 0.07
   AND (
        d.start_timestamp BETWEEN
            f.fire_detection_timestamp
                - INTERVAL 3 HOURS
            AND
            f.fire_detection_timestamp
                + INTERVAL 3 HOURS
        OR
        (
            d.start_timestamp
                <= f.fire_detection_timestamp
            AND (
                d.end_timestamp IS NULL
                OR
                d.end_timestamp
                    >= f.fire_detection_timestamp
            )
        )
   )

GROUP BY
    f.fire_detection_id,
    f.osm_way_id;


--Resultado

--he configurado el calculo del nivel de impacto combinando distancia e incidencias activas
CREATE OR REPLACE TEMP VIEW gold_fire_road_source AS

SELECT

    f.fire_detection_id,
    f.cluster_id,
    f.fire_detection_timestamp,
    f.fire_latitude,
    f.fire_longitude,
    f.fire_radiative_power,
    f.osm_way_id,
    f.road_reference,
    f.road_name,
    f.road_classification,
    f.road_latitude,
    f.road_longitude,
    f.distance_fire_road_km,
    f.max_speed_kmh,
    f.lanes_count,
    f.is_oneway,
    f.pavement_surface,
    f.has_bridge,
    f.has_tunnel,
    COALESCE(
        t.traffic_incident_count,
        0
    ) AS traffic_incident_count,
    COALESCE(
        t.active_traffic_incident_count,
        0
    ) AS active_traffic_incident_count,
    COALESCE(
        t.max_traffic_severity,
        'NORMAL'
    ) AS max_traffic_severity,
    CASE
        WHEN COALESCE(
            t.active_traffic_incident_count,
            0
        ) > 0
        AND f.distance_fire_road_km <= 5
            THEN 'HIGH'
        WHEN f.distance_fire_road_km <= 5
            THEN 'MEDIUM'
        WHEN f.distance_fire_road_km <= 15
            THEN 'LOW'
        ELSE 'VERY_LOW'
    END AS road_impact_level,
    f.fire_source_file,
    f.osm_source_file,
    current_timestamp()
        AS gold_updated_at

FROM fire_road_matches f

LEFT JOIN road_traffic t

    ON f.fire_detection_id =
       t.fire_detection_id
   AND f.osm_way_id =
       t.osm_way_id;

--MERGE

--actualizacion incremental mediante merge sobre la clave compuesta de incendio y tramo viario
MERGE INTO gold_fire_road_impact AS target

USING gold_fire_road_source AS source

ON target.fire_detection_id =
       source.fire_detection_id
AND target.osm_way_id =
       source.osm_way_id

WHEN MATCHED THEN UPDATE SET

    target.cluster_id = source.cluster_id,
    target.fire_detection_timestamp = source.fire_detection_timestamp,
    target.fire_latitude = source.fire_latitude,
    target.fire_longitude = source.fire_longitude,
    target.fire_radiative_power = source.fire_radiative_power,
    target.road_reference = source.road_reference,
    target.road_name = source.road_name,
    target.road_classification = source.road_classification,
    target.road_latitude = source.road_latitude,
    target.road_longitude = source.road_longitude,
    target.distance_fire_road_km = source.distance_fire_road_km,
    target.max_speed_kmh = source.max_speed_kmh,
    target.lanes_count = source.lanes_count,
    target.is_oneway = source.is_oneway,
    target.pavement_surface = source.pavement_surface,
    target.has_bridge = source.has_bridge,
    target.has_tunnel = source.has_tunnel,
    target.traffic_incident_count = source.traffic_incident_count,
    target.active_traffic_incident_count = source.active_traffic_incident_count,
    target.max_traffic_severity = source.max_traffic_severity,
    target.road_impact_level = source.road_impact_level,
    target.fire_source_file = source.fire_source_file,
    target.osm_source_file = source.osm_source_file,
    target.gold_updated_at = source.gold_updated_at


WHEN NOT MATCHED THEN INSERT (
    fire_detection_id,
    cluster_id,
    fire_detection_timestamp,
    fire_latitude,
    fire_longitude,
    fire_radiative_power,
    osm_way_id,
    road_reference,
    road_name,
    road_classification,
    road_latitude,
    road_longitude,
    distance_fire_road_km,
    max_speed_kmh,
    lanes_count,
    is_oneway,
    pavement_surface,
    has_bridge,
    has_tunnel,
    traffic_incident_count,
    active_traffic_incident_count,
    max_traffic_severity,
    road_impact_level,
    fire_source_file,
    osm_source_file,
    gold_updated_at
)

VALUES (
    source.fire_detection_id,
    source.cluster_id,
    source.fire_detection_timestamp,
    source.fire_latitude,
    source.fire_longitude,
    source.fire_radiative_power,
    source.osm_way_id,
    source.road_reference,
    source.road_name,
    source.road_classification,
    source.road_latitude,
    source.road_longitude,
    source.distance_fire_road_km,
    source.max_speed_kmh,
    source.lanes_count,
    source.is_oneway,
    source.pavement_surface,
    source.has_bridge,
    source.has_tunnel,
    source.traffic_incident_count,
    source.active_traffic_incident_count,
    source.max_traffic_severity,
    source.road_impact_level,
    source.fire_source_file,
    source.osm_source_file,
    source.gold_updated_at
);